In [ ]:
!pip install medmnist timm torch torchvision scikit-learn -q

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.npz'):
            print(os.path.join(root, f))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.amp import autocast, GradScaler
import torchvision.transforms as transforms
import timm
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from pathlib import Path
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)

In [ ]:
TASKS = [
    "pathmnist", "dermamnist", "octmnist", "pneumoniamnist",
    "retinamnist", "breastmnist", "bloodmnist", "tissuemnist",
    "organamnist", "organcmnist", "organsmnist"
]

TASK_N_CLASSES = {
    "pathmnist": 9,
    "dermamnist": 7,
    "octmnist": 4,
    "pneumoniamnist": 2,
    "retinamnist": 5,
    "breastmnist": 2,
    "bloodmnist": 8,
    "tissuemnist": 8,
    "organamnist": 11,
    "organcmnist": 11,
    "organsmnist": 11,
}

In [ ]:
IMG_SIZE   = 64
EPOCHS     = 10
BACKBONE   = "convnext_tiny" 
PRETRAINED = True
BATCH_SIZE = 256
LR         = 1e-3


# EXPERIMENT TOGGLES
USE_WEIGHTED_SAMPLER = False
USE_LABEL_SMOOTHING  = False

ws_str = "WS_ON" if USE_WEIGHTED_SAMPLER else "WS_OFF"
ls_str = "LS_ON" if USE_LABEL_SMOOTHING else "LS_OFF"
RUN_NAME = f"{BACKBONE}_{ws_str}_{ls_str}"
print(f"Run config: {RUN_NAME}")

In [ ]:
# Do NOT use this to download data at native 64px resolution.
# Training on the MedMNIST API's 64px images while evaluating on the
# Kaggle 28px test set causes a resolution domain shift that collapses
# leaderboard performance despite high validation scores.
# All data must be loaded from the 28px competition files and upscaled
# consistently in the transform pipeline (see cell below).
# 

# import medmnist
# from medmnist import INFO
# import os

# os.makedirs("/kaggle/working/data64", exist_ok=True)

# for task in TASKS:
#     info = INFO[task]
#     DataClass = getattr(medmnist, info["python_class"])

#     _ = DataClass(split="train", download=True, size=IMG_SIZE, root="/kaggle/working/data64")
#     _ = DataClass(split="val",   download=True, size=IMG_SIZE, root="/kaggle/working/data64")

In [ ]:
def load_split_from_dir(task: str, split: str, size: int, npz_dir: Path):
    suffix = "" if size == 28 else f"_{size}"
    path = npz_dir / f"{task}{suffix}.npz"
    z = np.load(path)
    images = z[f"{split}_images"]
    labels = z[f"{split}_labels"].squeeze().astype(np.int64)
    return images, labels

class MedMNISTTask(Dataset):
    def __init__(self, task: str, split: str, img_size: int, npz_dir: Path, transform=None):
        self.task_idx = TASKS.index(task)
        self.transform = transform
        self.images, self.labels = load_split_from_dir(task, split, size=img_size, npz_dir=npz_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]

        if img.ndim == 2:                          
            img = np.stack([img, img, img], axis=-1)
        elif img.shape[-1] == 1:                   
            img = np.concatenate([img, img, img], axis=-1)

        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx], self.task_idx

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import ConcatDataset, DataLoader
from pathlib import Path

# Point ALL data splits strictly to the Kaggle 28x28 folder
# Change this path if running outside Kaggle
KAGGLE_DIR = Path("/kaggle/input/competitions/tensor-reloaded-multi-task-med-mnist/data")
interpolation_mode = transforms.InterpolationMode.BICUBIC

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=interpolation_mode, antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=10,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
    ),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=interpolation_mode, antialias=True),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

test_transform_64 = val_transform 

# Load the 28px files for ALL splits, forcing them through the 64px bicubic stretch
train_ds = ConcatDataset([MedMNISTTask(t, "train", img_size=28, npz_dir=KAGGLE_DIR, transform=train_transform) for t in TASKS])
val_ds   = ConcatDataset([MedMNISTTask(t, "val",   img_size=28, npz_dir=KAGGLE_DIR, transform=val_transform)   for t in TASKS])
test_ds  = ConcatDataset([MedMNISTTask(t, "test",  img_size=28, npz_dir=KAGGLE_DIR, transform=test_transform_64) for t in TASKS])

weights = []
for dataset in train_ds.datasets:
    n = len(dataset)
    w = 1.0 / n
    weights.extend([w] * n)

from torch.utils.data import WeightedRandomSampler

if USE_WEIGHTED_SAMPLER:
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True, drop_last=True)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, drop_last=False)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, backbone_name, task_n_classes, pretrained=True):
        super().__init__()

        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        feat_dim = self.backbone.num_features
        print(f"Backbone: {backbone_name} | Feature dim: {feat_dim} | Pretrained: {pretrained}")

        self.heads = nn.ModuleDict({
            task: nn.Linear(feat_dim, n_cls)
            for task, n_cls in task_n_classes.items()
        })

    def forward(self, x, task_name):
        features = self.backbone(x)
        return self.heads[task_name](features)


model = MultiTaskModel(BACKBONE, TASK_N_CLASSES, pretrained=PRETRAINED).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
if USE_LABEL_SMOOTHING:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
else:
    criterion = nn.CrossEntropyLoss()
    
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    total_steps=EPOCHS * len(train_loader)
)
scaler = GradScaler("cuda")

def harmonic_mean(values):
    values = [v for v in values if v > 0]
    if not values:
        return 0.0
    return len(values) / sum(1.0 / v for v in values)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    task_preds   = {t: [] for t in TASKS}
    task_targets = {t: [] for t in TASKS}

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels, task_idxs in loader:
            images = images.to(device)
            labels = labels.to(device)

            with autocast("cuda"):
                features = model.backbone(images)

            loss = torch.tensor(0.0, device=device)
            n_tasks_in_batch = 0

            for tidx in torch.unique(task_idxs):
                task_name = TASKS[tidx.item()]
                mask = task_idxs == tidx

                if mask.sum() == 0:
                    continue

                task_features = features[mask]
                task_labels = labels[mask]


                with autocast("cuda"):
                    logits = model.heads[task_name](task_features)
                    task_loss = criterion(logits, task_labels)
                loss = loss + task_loss
                n_tasks_in_batch += 1

                preds = logits.argmax(dim=1).detach().cpu().numpy()
                task_preds[task_name].extend(preds)
                task_targets[task_name].extend(task_labels.cpu().numpy())

            if train and n_tasks_in_batch == 0:
                continue
            loss = loss / n_tasks_in_batch

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            total_loss += loss.item()

    task_f1 = {
        t: f1_score(task_targets[t], task_preds[t], average="macro", zero_division=0)
        for t in TASKS if task_preds[t]
    }
    hmean = harmonic_mean(list(task_f1.values()))
    return total_loss / len(loader), task_f1, hmean

In [ ]:
best_hmean = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1, train_hm = run_epoch(train_loader, train=True)
    val_loss,   val_f1,   val_hm   = run_epoch(val_loader,   train=False)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss, "train_hmean": train_hm,
        "val_loss": val_loss,     "val_hmean": val_hm,
        **{f"val_f1_{t}": val_f1.get(t, 0) for t in TASKS}
    })

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"  Train loss: {train_loss:.4f} | HMean F1: {train_hm:.4f}")
    print(f"  Val   loss: {val_loss:.4f} | HMean F1: {val_hm:.4f}")
    print(f"  Per-task F1: { {k: round(v,3) for k,v in val_f1.items()} }")

    if val_hm > best_hmean:
        best_hmean = val_hm
        torch.save(model.state_dict(), f"best_{RUN_NAME}.pth")
        print(f"  ** New best: {best_hmean:.4f} — saved **")

pd.DataFrame(history).to_csv(f"history_{RUN_NAME}.csv", index=False)
print(f"\nDone. Best val harmonic F1: {best_hmean:.4f}")

In [ ]:
model.load_state_dict(torch.load(f"best_{RUN_NAME}.pth"))
_, test_f1, test_hmean = run_epoch(test_loader, train=False)

print(f"\nTest Harmonic Mean F1: {test_hmean:.4f}")
print("\nPer-task test F1:")
for t, f in test_f1.items():
    print(f"  {t:15s}: {f:.4f}")

In [ ]:
model.load_state_dict(torch.load(f"best_{RUN_NAME}.pth"))
model.eval()

rows = []
global_id = 0
task_counters = {t: 0 for t in TASKS}

with torch.no_grad():
    for images, _, task_idxs in test_loader:
        images = images.to(device)

        features = model.backbone(images)

        for i in range(images.size(0)):
            task_name = TASKS[task_idxs[i].item()]
            logits = model.heads[task_name](features[i:i+1])
            pred = logits.argmax(dim=1).item()

            rows.append({
                "id": global_id,
                "id_image_in_task": task_counters[task_name],
                "task_name": task_name,
                "label": int(pred),
            })

            global_id += 1
            task_counters[task_name] += 1

In [ ]:
assert len(rows) == len(test_ds)

In [ ]:
sub = pd.DataFrame(rows)[["id", "id_image_in_task", "task_name", "label"]]
sub.to_csv(f"submission_{RUN_NAME}.csv", index=False)
print(f"Saved: {len(sub)} rows")
print(sub.head())